# Tutorial 1 — Clickbait detection with pretrained embeddings

**Goal:** build two baselines for clickbait detection and evaluate them properly.

1. TF-IDF + logistic regression (no pretrained model)
2. Frozen **pretrained sentence embeddings** + logistic regression

**Data:** English clickbait headlines from Chakraborty et al. (2016), *Stop Clickbait*, loaded from the Hugging Face Hub (`marksverdhei/clickbait_title_classification`).

**Runtime:** works on CPU. A GPU (*Runtime → Change runtime type → T4 GPU*) makes the encoding step faster.

In [ ]:
!pip -q install -U sentence-transformers datasets

## 1. Load and inspect the data

In [ ]:
from datasets import load_dataset
import pandas as pd

# English clickbait headlines (Chakraborty et al., 2016) — ~32k headlines, MIT licence
en = load_dataset("marksverdhei/clickbait_title_classification", split="train").to_pandas()
en = en.rename(columns={"title": "text", "clickbait": "label"})[["text", "label"]]
en = en.dropna().drop_duplicates("text").reset_index(drop=True)
print(en.shape)
print(en["label"].value_counts().rename({0: "not clickbait", 1: "clickbait"}))
en.sample(5, random_state=0)

In [ ]:
for lab, name in [(1, "CLICKBAIT"), (0, "NOT CLICKBAIT")]:
    print(f"--- {name} ---")
    for t in en[en.label == lab].text.sample(5, random_state=1):
        print("  ", t)

**Question:** before training anything, which surface cues do you notice in the clickbait headlines? (Numbers, “you”, questions, superlatives…)

## 2. Baseline A — TF-IDF + logistic regression

We use **stratified 5-fold cross-validation** and report mean ± standard deviation, not a single lucky split.

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

texts, y = en["text"].tolist(), en["label"].values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["f1_macro", "roc_auc", "accuracy"]

def report(name, res):
    row = {"model": name}
    for m in scoring:
        s = res[f"test_{m}"]
        row[m] = f"{s.mean():.4f} ± {s.std():.4f}"
    return row

tfidf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                      LogisticRegression(max_iter=2000))
res_tfidf = cross_validate(tfidf, texts, y, cv=cv, scoring=scoring)
results = [report("TF-IDF + LogReg", res_tfidf)]
pd.DataFrame(results)

## 3. Baseline B — pretrained sentence embeddings

A pretrained encoder turns each headline into one vector. We **freeze** the encoder (no training) and train only a logistic regression on top.

We use a *multilingual* model, so the same classifier can later be applied to Indonesian or Polish headlines (Tutorial 3).
Tokenization happens inside `encode()`: the model's own tokenizer converts the text into the IDs it was trained on.

In [ ]:
import time
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
t0 = time.time()
X = encoder.encode(texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
print("Embedding matrix:", X.shape, f"— {(time.time()-t0)/len(texts)*1000:.2f} ms per headline")

In [ ]:
res_emb = cross_validate(LogisticRegression(max_iter=2000), X, y, cv=cv, scoring=scoring)
results.append(report("Multilingual MiniLM embeddings + LogReg", res_emb))
pd.DataFrame(results)

**Discuss:** how big is the gap between A and B? Is the difference larger than the standard deviation across folds?

Note: ROC-AUC here is computed from predicted *probabilities*. Computing it from hard 0/1 predictions is a common mistake that gives a much lower, misleading number.

## 4. Error analysis — read the mistakes

In [ ]:
from sklearn.model_selection import cross_val_predict

proba = cross_val_predict(LogisticRegression(max_iter=2000), X, y, cv=cv, method="predict_proba")[:, 1]
en["p_clickbait"] = proba
en["pred"] = (proba >= 0.5).astype(int)
errors = en[en.pred != en.label].copy()
errors["confidence"] = (errors.p_clickbait - 0.5).abs()
print(f"{len(errors)} errors out of {len(en)}")
pd.set_option("display.max_colwidth", 120)
errors.sort_values("confidence", ascending=False).head(15)[["text", "label", "p_clickbait"]]

Look at the most confident errors. Are some of them **label noise** (the gold label looks wrong to you)? Are some genuinely ambiguous?

## 5. Try your own headlines

In [ ]:
clf = LogisticRegression(max_iter=2000).fit(X, y)
mine = [
    "You won't believe what this professor said in class today",
    "European Parliament approves new rules on artificial intelligence",
    "10 reasons why your code doesn't work",
    "Nie uwierzysz, co znaleziono w Wiśle!",          # Polish
    "Ini alasannya kenapa kamu harus minum air putih", # Indonesian
]
for t, p in zip(mine, clf.predict_proba(encoder.encode(mine, normalize_embeddings=True))[:, 1]):
    print(f"{p:.2f}  {t}")

## Exercises

1. Replace logistic regression with `xgboost.XGBClassifier` on the same embeddings. Does it help?
2. Swap the encoder for a stronger one (e.g. `sentence-transformers/all-mpnet-base-v2`, English-only). How much does F1 change, and how much slower is encoding?
3. Add simple style features (number of words, starts with a digit, contains “you”, ends with “?”) to TF-IDF and to the embeddings. Do they help the weak model, the strong one, or both?
4. Time the classifier's prediction **per headline**, separately from training and from encoding.

---
*Data:* Chakraborty, A., Paranjape, B., Kakarla, S., & Ganguly, N. (2016). *Stop Clickbait: Detecting and Preventing Clickbaits in Online News Media.* ASONAM.